# C4-4 YOLO One-Time Final Test

This dedicated surface keeps preflight separate from the explicit operator unlock. It does not select, train, or tune a candidate.

## 1. Operator paths and execution lock

In [ ]:
import os
import subprocess
from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()
OFFICIAL_PACKAGE_VALUE = os.environ.get("C4_4_OFFICIAL_PACKAGE")
DATASET_ROOT_VALUE = os.environ.get("C4_4_DATASET_ROOT")
EXPECTED_C4_4_COMMIT = os.environ.get("C4_4_EXPECTED_COMMIT")
DEVICE = os.environ.get("C4_4_DEVICE", "cuda")
RUN_FINAL_TEST = False

if not OFFICIAL_PACKAGE_VALUE or not DATASET_ROOT_VALUE or not EXPECTED_C4_4_COMMIT:
    raise RuntimeError("Set C4_4_OFFICIAL_PACKAGE, C4_4_DATASET_ROOT, and C4_4_EXPECTED_COMMIT.")
if len(EXPECTED_C4_4_COMMIT) != 40 or any(
    c not in "0123456789abcdef" for c in EXPECTED_C4_4_COMMIT
):
    raise ValueError("C4_4_EXPECTED_COMMIT must be a full lowercase Git SHA.")
OFFICIAL_PACKAGE = Path(OFFICIAL_PACKAGE_VALUE)
DATASET_ROOT = Path(DATASET_ROOT_VALUE)
if not OFFICIAL_PACKAGE.is_file():
    raise FileNotFoundError("Set C4_4_OFFICIAL_PACKAGE to the Official C4-2C ZIP.")
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError("Set C4_4_DATASET_ROOT to the derived dataset root.")

## 2. Locked environment and Git verification

In [ ]:
subprocess.run(["uv", "sync", "--locked"], cwd=REPOSITORY_ROOT, check=True)
subprocess.run(["git", "status", "--short"], cwd=REPOSITORY_ROOT, check=True)
actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_ROOT, text=True
).strip()
if actual_commit != EXPECTED_C4_4_COMMIT:
    raise RuntimeError("Checkout does not match the reviewed C4-4 execution commit.")

## 3. Frozen candidate, Official package, and Manifest-byte preflight

This command verifies the frozen lifecycle, all artifact identities, repository config, clean Git provenance, and Dataset Manifest SHA. It does not resolve CSV rows or open test images/labels.

In [ ]:
preflight_command = [
    "uv",
    "run",
    "--locked",
    "python",
    "-m",
    "pipelines.evaluate_yolo_final_test",
    "--repository-root",
    str(REPOSITORY_ROOT),
    "--official-package",
    str(OFFICIAL_PACKAGE),
    "--dataset",
    str(DATASET_ROOT),
    "--device",
    DEVICE,
]
subprocess.run(preflight_command, cwd=REPOSITORY_ROOT, check=True)
print("READY_FOR_FINAL_TEST; FINAL TEST remains SEALED_NOT_USED")

## 4. Explicit one-time operator unlock and evidence export

Review the preflight output and set `RUN_FINAL_TEST = True` only for the authorized final run. The output namespace fails closed if it already exists.

In [ ]:
if RUN_FINAL_TEST is not True:
    raise RuntimeError("Final test remains sealed; explicit operator unlock was not granted.")

execution_command = [*preflight_command, "--confirm-final-test"]
subprocess.run(execution_command, cwd=REPOSITORY_ROOT, check=True)

## 5. Closure boundary

The exported C4-4 result is report-only. Do not use final-test metrics for retraining, threshold changes, checkpoint selection, or candidate promotion.